# Notebook 3 — Production Model: LightGBM Full Retrain & 3-Hour Forecast

**Pipeline summary:**
1. Reload and validate the full dataset (all 30 days)
2. Rebuild the exact feature engineering from Notebook 1 on the **entire** series
3. Retrain LightGBM on 100% of available data — no held-out test set
4. Persist the model + metadata to disk (`lgb_production.pkl`)
5. Generate a 2160-step (3-hour) forecast starting from the last observed timestamp
6. Export the forecast to `forecast_output.csv`
7. Visualise the forecast with confidence context

> **Why LightGBM?**  
> Across the ML leaderboard in Notebook 1, LightGBM consistently achieved the  
> lowest SMAPE on the recursive 3-hour forecast — with the fastest training time  
> of all gradient boosters on this 500k-row dataset.


## Cell 0 — Environment & Constants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pickle
import json
import os
import warnings
from datetime import timedelta

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.size': 11,
})

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Domain constants ──────────────────────────────────────────────────────────
FREQ_SEC         = 5
HOURLY           = int(3600  / FREQ_SEC)   # 720 steps
DAILY            = int(86400 / FREQ_SEC)   # 17 280 steps
FORECAST_HORIZON = int(3 * 3600 / FREQ_SEC)  # 2 160 steps = 3 hours

print(f"Forecast horizon : {FORECAST_HORIZON} steps")
print(f"Step resolution  : {FREQ_SEC} seconds")
print(f"Horizon duration : {FORECAST_HORIZON * FREQ_SEC / 3600:.1f} hours")


## Cell 1 — Load & Validate Full Dataset

In [ ]:
df_raw = pd.read_excel("problem_data.xlsx")

print(f"Raw shape        : {df_raw.shape}")
print(f"Columns          : {df_raw.columns.tolist()}")
print(f"Dtypes:\n{df_raw.dtypes}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")


In [ ]:
# ── Cleaning (mirrors EDA notebook exactly) ───────────────────────────────────
df_clean = (
    df_raw
    .dropna(subset=["Reading"])
    .drop_duplicates(subset=["timestamp"], keep="first")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# ── Resample to strict 5-second grid (fills gaps with 0 = outage) ─────────────
df_5s = (
    df_clean
    .set_index("timestamp")[["Reading"]]
    .resample("5s").sum()          # sum collapses duplicates; 0 on missing
    .reset_index()
)

# Sanity checks
total_steps  = len(df_5s)
span_days    = (df_5s["timestamp"].iloc[-1] - df_5s["timestamp"].iloc[0]).days
outage_rate  = (df_5s["Reading"] == 0).mean()
global_mean  = df_5s["Reading"].mean()
global_std   = df_5s["Reading"].std()

print(f"\nFull resampled series")
print(f"  Total steps   : {total_steps:,}")
print(f"  Date range    : {df_5s['timestamp'].iloc[0]}  →  {df_5s['timestamp'].iloc[-1]}")
print(f"  Span          : {span_days} days")
print(f"  Outage rate   : {outage_rate*100:.2f}%")
print(f"  Global mean   : {global_mean:.4f}")
print(f"  Global std    : {global_std:.4f}")

GLOBAL_MEAN = global_mean
GLOBAL_STD  = global_std

# Last observed timestamp — forecasts begin immediately after this
LAST_TIMESTAMP = df_5s["timestamp"].iloc[-1]
FORECAST_START = LAST_TIMESTAMP + pd.Timedelta(seconds=FREQ_SEC)
print(f"\nLast observed   : {LAST_TIMESTAMP}")
print(f"Forecast starts : {FORECAST_START}")


## Cell 2 — Feature Engineering (Full Dataset)

Identical feature set to Notebook 1. All features are strictly backward-looking.  
Running on the full 30-day series — the warm-up rows dropped by `max_lag` are  
negligible (<0.5% of data) and are excluded cleanly.


In [ ]:
def build_features(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    """
    Build the full feature matrix from a time-indexed DataFrame.
    Identical implementation to Notebook 1 — single source of truth.
    All rolling/shift operations are strictly backward-looking (no leakage).
    """
    df = df.copy().set_index("timestamp")
    y  = df["Reading"]

    feats = pd.DataFrame(index=df.index)

    # ── 1. Lag features ───────────────────────────────────────────────────────
    LAG_STEPS = [1, 2, 3, 5, 10, 12, 30, 60, 120, 360, 720, 1440, FORECAST_HORIZON]
    for lag in LAG_STEPS:
        feats[f"lag_{lag}"] = y.shift(lag)

    # ── 2. Rolling statistics ─────────────────────────────────────────────────
    for win in [12, 60, 360, 720]:
        r = y.shift(1).rolling(window=win, min_periods=max(1, win // 4))
        feats[f"roll_mean_{win}"]  = r.mean()
        feats[f"roll_std_{win}"]   = r.std().fillna(0)
        feats[f"roll_max_{win}"]   = r.max()
        feats[f"roll_min_{win}"]   = r.min()
        feats[f"roll_range_{win}"] = feats[f"roll_max_{win}"] - feats[f"roll_min_{win}"]

    # ── 3. Exponentially Weighted features ────────────────────────────────────
    for span in [12, 60, 360]:
        feats[f"ewm_mean_{span}"] = y.shift(1).ewm(span=span, adjust=False).mean()
        feats[f"ewm_std_{span}"]  = y.shift(1).ewm(span=span, adjust=False).std().fillna(0)

    # ── 4. Diff / rate-of-change ──────────────────────────────────────────────
    feats["diff_1"]  = y.diff(1).shift(1)
    feats["diff_12"] = y.diff(12).shift(1)
    feats["diff_60"] = y.diff(60).shift(1)

    # ── 5. Cyclical time encodings ────────────────────────────────────────────
    ts = df.index
    feats["hour_sin"]   = np.sin(2 * np.pi * ts.hour / 24)
    feats["hour_cos"]   = np.cos(2 * np.pi * ts.hour / 24)
    feats["minute_sin"] = np.sin(2 * np.pi * ts.minute / 60)
    feats["minute_cos"] = np.cos(2 * np.pi * ts.minute / 60)
    feats["dow_sin"]    = np.sin(2 * np.pi * ts.dayofweek / 7)
    feats["dow_cos"]    = np.cos(2 * np.pi * ts.dayofweek / 7)

    # ── 6. Outage indicators ──────────────────────────────────────────────────
    feats["is_zero"] = (y.shift(1) == 0).astype(int)
    feats["zero_run_length"] = (
        feats["is_zero"]
        .groupby((feats["is_zero"] != feats["is_zero"].shift()).cumsum())
        .cumsum()
    )
    feats["steps_since_nonzero"] = feats["zero_run_length"]

    # ── 7. Local trend proxy ──────────────────────────────────────────────────
    feats["local_trend"] = (
        (feats["roll_mean_60"] - feats["roll_mean_12"]) / (GLOBAL_STD + 1e-8)
    )

    # ── Target ────────────────────────────────────────────────────────────────
    feats["target"] = y.values

    if is_train:
        max_lag = max(LAG_STEPS)
        feats   = feats.iloc[max_lag:].dropna()

    return feats.reset_index()


print("Building feature matrix on full dataset...")
full_feats = build_features(df_5s, is_train=True)

FEATURE_COLS = [c for c in full_feats.columns if c not in ("timestamp", "target")]

print(f"  Full feature matrix : {full_feats.shape}")
print(f"  Feature count       : {len(FEATURE_COLS)}")
print(f"  Missing values      : {full_feats[FEATURE_COLS].isna().sum().sum()}")


## Cell 3 — Retrain LightGBM on 100% of Data

In production, we **do not hold out a test set** — we use all available  
signal to maximise model quality. The hyperparameters are carried directly  
from Notebook 1 (validated on the 7-day hold-out).

The only change: `n_estimators` is fixed (no early stopping without a  
validation set). We use the `best_iteration_` from Notebook 1 training  
plus a small buffer (+10%) to ensure convergence.


In [ ]:
from lightgbm import LGBMRegressor
import lightgbm as lgb

X_full = full_feats[FEATURE_COLS].values.astype(np.float32)
y_full = full_feats["target"].values.astype(np.float32)

print(f"Training on {len(X_full):,} samples with {len(FEATURE_COLS)} features")

# ── Hyperparameters (identical to Notebook 1 validated config) ────────────────
# n_estimators fixed — use best_iteration from NB1 + 10% buffer.
# If you have the value, replace N_ESTIMATORS_FINAL below.
N_ESTIMATORS_FINAL = 600   # ← replace with int(nb1_best_iteration * 1.1) if known

lgb_prod = LGBMRegressor(
    n_estimators      = N_ESTIMATORS_FINAL,
    num_leaves        = 63,
    max_depth         = -1,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = RANDOM_SEED,
    n_jobs            = -1,
    verbosity         = -1,
)

lgb_prod.fit(
    X_full, y_full,
    callbacks=[lgb.log_evaluation(100)],
)

print(f"\nLightGBM production model trained on {len(X_full):,} rows.")
print(f"  Trees            : {lgb_prod.n_estimators_}")
print(f"  Features used    : {lgb_prod.n_features_in_}")


## Cell 4 — Save Model & Metadata

In [ ]:
import pickle, json
from datetime import datetime

MODEL_PATH    = "lgb_production.pkl"
METADATA_PATH = "lgb_production_metadata.json"

# ── Pickle the fitted model ───────────────────────────────────────────────────
with open(MODEL_PATH, "wb") as f:
    pickle.dump(lgb_prod, f)

print(f"Model saved → {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1024:.1f} KB)")

# ── Save companion metadata (everything needed to reproduce inference) ─────────
metadata = {
    "model_type"         : "LGBMRegressor",
    "trained_at"         : datetime.now().isoformat(),
    "training_rows"      : int(len(X_full)),
    "n_estimators"       : int(lgb_prod.n_estimators_),
    "feature_cols"       : FEATURE_COLS,
    "freq_sec"           : FREQ_SEC,
    "forecast_horizon"   : FORECAST_HORIZON,
    "global_mean"        : float(GLOBAL_MEAN),
    "global_std"         : float(GLOBAL_STD),
    "last_timestamp"     : str(LAST_TIMESTAMP),
    "forecast_start"     : str(FORECAST_START),
    "data_start"         : str(df_5s["timestamp"].iloc[0]),
    "data_end"           : str(df_5s["timestamp"].iloc[-1]),
    "total_steps"        : int(len(df_5s)),
    "hyperparameters": {
        "num_leaves"        : 63,
        "learning_rate"     : 0.05,
        "subsample"         : 0.8,
        "colsample_bytree"  : 0.8,
        "min_child_samples" : 20,
        "reg_alpha"         : 0.1,
        "reg_lambda"        : 1.0,
    },
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved → {METADATA_PATH}")
print(f"\nMetadata preview:")
for k, v in metadata.items():
    if k not in ("feature_cols",):
        print(f"  {k:25s}: {v}")
print(f"  {'feature_cols':25s}: [{FEATURE_COLS[0]}, ..., {FEATURE_COLS[-1]}]  ({len(FEATURE_COLS)} total)")


## Cell 5 — Verify Model Round-Trip (Load & Smoke Test)

In [ ]:
# ── Reload from disk to confirm serialisation is clean ────────────────────────
with open(MODEL_PATH, "rb") as f:
    lgb_loaded = pickle.load(f)

# Quick smoke test on first 1000 rows
smoke_preds = lgb_loaded.predict(X_full[:1000])
assert smoke_preds.shape == (1000,), "Shape mismatch after reload"
assert not np.any(np.isnan(smoke_preds)), "NaN in smoke-test predictions"

print(f"Round-trip verification passed.")
print(f"  Loaded model type : {type(lgb_loaded).__name__}")
print(f"  Smoke test preds  : min={smoke_preds.min():.4f}  max={smoke_preds.max():.4f}  mean={smoke_preds.mean():.4f}")
print(f"  Actual range      : min={y_full[:1000].min():.4f}  max={y_full[:1000].max():.4f}")


## Cell 6 — Recursive Forecaster for Production Inference

The `RecursiveForecaster` maintains a rolling in-memory buffer of recent  
values and incrementally updates all stateful features (lags, rolling stats,  
EWM) at each prediction step — exactly matching how features were built  
during training. This guarantees zero train/inference skew.


In [ ]:
from collections import deque

class RecursiveForecaster:
    """
    Production-grade recursive forecaster for LightGBM IoT models.

    Maintains incremental state for:
      - Lag features   : O(1) via ring buffer
      - Rolling stats  : O(window) via ring buffer slices
      - EWM            : O(1) via running mean/variance update
      - Cyclical time  : O(1) computed from timestamp
      - Outage flags   : O(1) via zero-run counter

    Parameters
    ----------
    model        : Fitted sklearn-compatible regressor (.predict)
    feature_cols : List of feature column names (same order as training)
    history_vals : np.array — recent actual readings (≥ max_lag length)
    history_ts   : pd.DatetimeIndex matching history_vals
    freq         : Step timedelta (default 5 s)
    global_std   : float — used for local_trend normalisation
    """

    LAG_STEPS    = [1, 2, 3, 5, 10, 12, 30, 60, 120, 360, 720, 1440, FORECAST_HORIZON]
    ROLL_WINDOWS = [12, 60, 360, 720]
    EWM_SPANS    = [12, 60, 360]
    DIFF_STEPS   = [1, 12, 60]

    def __init__(self, model, feature_cols, history_vals, history_ts,
                 freq=pd.Timedelta("5s"), global_std=1.0):
        self.model        = model
        self.feature_cols = feature_cols
        self.freq         = freq
        self.global_std   = global_std

        max_lag      = max(self.LAG_STEPS)
        max_roll     = max(self.ROLL_WINDOWS)
        self.buf_len = max(max_lag, max_roll) + 50   # generous buffer

        self.buf_vals = deque(history_vals[-self.buf_len:], maxlen=self.buf_len)
        self.buf_ts   = deque(list(history_ts[-self.buf_len:]), maxlen=self.buf_len)

        # Initialise EWM from history
        self._alpha    = {s: 2 / (s + 1) for s in self.EWM_SPANS}
        init_arr       = np.array(self.buf_vals)
        self._ewm_mean = {s: float(np.mean(init_arr)) for s in self.EWM_SPANS}
        self._ewm_var  = {s: float(np.var(init_arr))  for s in self.EWM_SPANS}

    def _update_ewm(self, val):
        for s in self.EWM_SPANS:
            a = self._alpha[s]
            old_mean = self._ewm_mean[s]
            self._ewm_mean[s] = a * val + (1 - a) * old_mean
            self._ewm_var[s]  = (1 - a) * (self._ewm_var[s] +
                                 a * (val - old_mean) ** 2)

    def _build_row(self, next_ts: pd.Timestamp) -> np.ndarray:
        buf = np.array(self.buf_vals)
        row = {}

        # Lags
        for lag in self.LAG_STEPS:
            row[f"lag_{lag}"] = float(buf[-lag]) if len(buf) >= lag else 0.0

        # Rolling statistics
        for win in self.ROLL_WINDOWS:
            w = buf[-win:] if len(buf) >= win else buf
            row[f"roll_mean_{win}"]  = float(np.mean(w))
            row[f"roll_std_{win}"]   = float(np.std(w))
            row[f"roll_max_{win}"]   = float(np.max(w))
            row[f"roll_min_{win}"]   = float(np.min(w))
            row[f"roll_range_{win}"] = row[f"roll_max_{win}"] - row[f"roll_min_{win}"]

        # EWM
        for s in self.EWM_SPANS:
            row[f"ewm_mean_{s}"] = self._ewm_mean[s]
            row[f"ewm_std_{s}"]  = float(np.sqrt(max(self._ewm_var[s], 0)))

        # Diffs
        for d in self.DIFF_STEPS:
            row[f"diff_{d}"] = float(buf[-1] - buf[-d-1]) if len(buf) > d else 0.0

        # Cyclical time
        row["hour_sin"]   = np.sin(2 * np.pi * next_ts.hour   / 24)
        row["hour_cos"]   = np.cos(2 * np.pi * next_ts.hour   / 24)
        row["minute_sin"] = np.sin(2 * np.pi * next_ts.minute / 60)
        row["minute_cos"] = np.cos(2 * np.pi * next_ts.minute / 60)
        row["dow_sin"]    = np.sin(2 * np.pi * next_ts.dayofweek / 7)
        row["dow_cos"]    = np.cos(2 * np.pi * next_ts.dayofweek / 7)

        # Outage indicators
        zero_run = 0
        for v in reversed(list(self.buf_vals)):
            if v == 0: zero_run += 1
            else: break
        row["is_zero"]             = int(buf[-1] == 0)
        row["zero_run_length"]     = zero_run
        row["steps_since_nonzero"] = zero_run

        # Local trend
        row["local_trend"] = (row["roll_mean_60"] - row["roll_mean_12"]) / (self.global_std + 1e-8)

        return np.array([row[c] for c in self.feature_cols], dtype=np.float32)

    def forecast(self, n_steps: int, start_ts: pd.Timestamp,
                 clip_negative: bool = True) -> tuple:
        """
        Generate n_steps recursive predictions.

        Returns
        -------
        timestamps : pd.DatetimeIndex of forecast timestamps
        predictions: np.array of predicted readings
        """
        preds      = np.empty(n_steps, dtype=np.float32)
        timestamps = pd.date_range(start=start_ts, periods=n_steps,
                                   freq=f"{FREQ_SEC}s")
        cur_ts = start_ts

        for i in range(n_steps):
            x_row  = self._build_row(cur_ts).reshape(1, -1)
            pred   = float(self.model.predict(x_row)[0])
            if clip_negative:
                pred = max(pred, 0.0)
            preds[i] = pred
            self._update_ewm(pred)
            self.buf_vals.append(pred)
            self.buf_ts.append(cur_ts)
            cur_ts += self.freq

        return timestamps, preds


print("RecursiveForecaster class defined.")


## Cell 7 — Generate 3-Hour Production Forecast

In [ ]:
import time

# ── Seed the forecaster with the full historical series ───────────────────────
history_vals = df_5s["Reading"].values.astype(np.float32)
history_ts   = pd.DatetimeIndex(df_5s["timestamp"].values)

forecaster = RecursiveForecaster(
    model        = lgb_loaded,          # loaded from disk — proves round-trip
    feature_cols = FEATURE_COLS,
    history_vals = history_vals,
    history_ts   = history_ts,
    freq         = pd.Timedelta(seconds=FREQ_SEC),
    global_std   = GLOBAL_STD,
)

print(f"Forecaster seeded with {len(history_vals):,} historical observations.")
print(f"Generating {FORECAST_HORIZON}-step forecast starting at {FORECAST_START} ...")

t0 = time.perf_counter()
forecast_timestamps, forecast_values = forecaster.forecast(
    n_steps  = FORECAST_HORIZON,
    start_ts = FORECAST_START,
)
elapsed = time.perf_counter() - t0

print(f"\nForecast complete in {elapsed:.2f}s  ({elapsed/FORECAST_HORIZON*1000:.2f} ms/step)")
print(f"  Steps generated : {len(forecast_values):,}")
print(f"  Horizon end     : {forecast_timestamps[-1]}")
print(f"\nForecast statistics:")
print(f"  Min             : {forecast_values.min():.4f}")
print(f"  Max             : {forecast_values.max():.4f}")
print(f"  Mean            : {forecast_values.mean():.4f}")
print(f"  Std             : {forecast_values.std():.4f}")
print(f"  Zero steps      : {(forecast_values == 0).sum()}  ({(forecast_values==0).mean()*100:.1f}%)")


## Cell 8 — Export Forecast to CSV

In [ ]:
FORECAST_PATH = "forecast_output.csv"

forecast_df = pd.DataFrame({
    "timestamp"       : forecast_timestamps,
    "forecast_reading": forecast_values.round(6),
    "step"            : np.arange(1, FORECAST_HORIZON + 1),
    "minutes_ahead"   : (np.arange(FORECAST_HORIZON) * FREQ_SEC / 60).round(2),
})

forecast_df.to_csv(FORECAST_PATH, index=False)

print(f"Forecast exported → {FORECAST_PATH}")
print(f"  Rows     : {len(forecast_df):,}")
print(f"  Columns  : {forecast_df.columns.tolist()}")
print()
print(forecast_df.head(10).to_string(index=False))
print("...")
print(forecast_df.tail(5).to_string(index=False))


## Cell 9 — Feature Importance (Production Model)

In [ ]:
importances = pd.Series(lgb_prod.feature_importances_, index=FEATURE_COLS)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#2ecc71" if i < 5 else "#3498db" for i in range(len(top20))]
top20.plot.barh(ax=ax, color=colors, edgecolor="white")
ax.set_title("LightGBM Production Model — Top 20 Feature Importances", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (split gain)")
ax.invert_yaxis()

# Annotate top-5
for i, (name, val) in enumerate(top20.head(5).items()):
    ax.text(val + top20.max()*0.01, i, f"{val:.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → feature_importance.png")


## Cell 10 — Forecast Visualisation

In [ ]:
# ── Plot: last 6 hours of actuals + 3-hour forecast ──────────────────────────
CONTEXT_STEPS = 6 * HOURLY   # 6 hours of history for visual context

history_context = df_5s.iloc[-CONTEXT_STEPS:].copy()

fig, axes = plt.subplots(2, 1, figsize=(16, 9),
                          gridspec_kw={"height_ratios": [3, 1.2]})

# ── Panel 1: Full view ────────────────────────────────────────────────────────
ax = axes[0]

ax.plot(history_context["timestamp"], history_context["Reading"],
        label="Historical (last 6 h)", color="steelblue",
        linewidth=0.9, alpha=0.85)

ax.plot(forecast_timestamps, forecast_values,
        label="Forecast (next 3 h)", color="tomato",
        linewidth=1.2, alpha=0.95)

ax.axvline(LAST_TIMESTAMP, color="black", linestyle="--",
           linewidth=1.2, label="Forecast origin")

ax.fill_between(forecast_timestamps, 0, forecast_values,
                color="tomato", alpha=0.08)

ax.set_title("LightGBM Production Forecast — Historical Context + 3-Hour Ahead",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Reading")
ax.legend(loc="upper left", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# ── Panel 2: Forecast close-up with 30-min rolling mean ──────────────────────
ax2 = axes[1]
ax2.plot(forecast_timestamps, forecast_values,
         color="tomato", linewidth=0.8, alpha=0.6, label="Raw forecast")

# 30-min rolling mean (360 steps) for trend readability
roll_mean = pd.Series(forecast_values).rolling(360, min_periods=1).mean().values
ax2.plot(forecast_timestamps, roll_mean,
         color="darkred", linewidth=1.5, label="30-min rolling mean")

ax2.set_title("Forecast Detail — Raw Steps + 30-min Trend", fontsize=11)
ax2.set_ylabel("Reading")
ax2.set_xlabel("Timestamp")
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30]))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig("forecast_plot.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → forecast_plot.png")


## Cell 11 — Hourly Breakdown & Summary Statistics

In [ ]:
forecast_df["hour_block"] = pd.cut(
    forecast_df["minutes_ahead"],
    bins   = [0, 60, 120, 180],
    labels = ["Hour 1 (0–60 min)", "Hour 2 (60–120 min)", "Hour 3 (120–180 min)"],
    right  = True,
)

hourly_summary = (
    forecast_df.groupby("hour_block", observed=True)["forecast_reading"]
    .agg(["mean", "std", "min", "max", "median"])
    .round(4)
)
hourly_summary.columns = ["Mean", "Std", "Min", "Max", "Median"]

print("Forecast summary by hour block:")
print(hourly_summary.to_string())


In [ ]:
# ── Bar chart of hourly mean forecast ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar: mean per hour block
hourly_summary["Mean"].plot.bar(ax=axes[0], color=["#3498db","#e67e22","#e74c3c"],
                                 edgecolor="white", width=0.5)
axes[0].set_title("Mean Forecast Reading by Hour Block", fontsize=12)
axes[0].set_ylabel("Mean Reading")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(hourly_summary["Mean"]):
    axes[0].text(i, v + hourly_summary["Mean"].max()*0.01,
                 f"{v:.2f}", ha="center", fontsize=10)

# Line: 5-min rolling mean over full horizon
mins       = forecast_df["minutes_ahead"].values
roll_5min  = pd.Series(forecast_values).rolling(60, min_periods=1).mean().values
axes[1].plot(mins, forecast_values, alpha=0.25, color="tomato", linewidth=0.5)
axes[1].plot(mins, roll_5min,       color="darkred", linewidth=1.8, label="5-min rolling mean")
axes[1].axvline(60,  color="gray", linestyle="--", linewidth=1, alpha=0.7)
axes[1].axvline(120, color="gray", linestyle="--", linewidth=1, alpha=0.7)
axes[1].text(30,  forecast_values.max()*0.98, "Hr 1", ha="center", color="gray", fontsize=9)
axes[1].text(90,  forecast_values.max()*0.98, "Hr 2", ha="center", color="gray", fontsize=9)
axes[1].text(150, forecast_values.max()*0.98, "Hr 3", ha="center", color="gray", fontsize=9)
axes[1].set_title("Forecast Trend Over 3-Hour Horizon", fontsize=12)
axes[1].set_xlabel("Minutes Ahead")
axes[1].set_ylabel("Reading")
axes[1].legend()

plt.tight_layout()
plt.savefig("forecast_hourly_breakdown.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → forecast_hourly_breakdown.png")


## Cell 12 — Output Artifact Inventory

In [ ]:
artifacts = {
    "lgb_production.pkl"            : "Fitted LightGBM model (pickle)",
    "lgb_production_metadata.json"  : "Model metadata, hyperparams, feature list",
    "forecast_output.csv"           : "2160-step forecast with timestamps",
    "forecast_plot.png"             : "Historical + forecast visualisation",
    "feature_importance.png"        : "Top-20 feature importances",
    "forecast_hourly_breakdown.png" : "Hourly summary chart",
}

print("\n╔══════════════════════════════════════════════════════════╗")
print("║              OUTPUT ARTIFACTS                            ║")
print("╠══════════════════════════════════════════════════════════╣")
for fname, desc in artifacts.items():
    exists = "✓" if os.path.exists(fname) else "✗ MISSING"
    size   = f"{os.path.getsize(fname)/1024:.1f} KB" if os.path.exists(fname) else ""
    print(f"║  {exists}  {fname:<38s} {size:<10s}║")
print("╚══════════════════════════════════════════════════════════╝")

# Quick reload demo — proves the saved model is self-contained
with open("lgb_production.pkl", "rb") as f:
    _m = pickle.load(f)
with open("lgb_production_metadata.json") as f:
    _meta = json.load(f)

print(f"\nModel reload check  : OK — {_m.n_estimators_} trees, {_meta['training_rows']:,} training rows")
print(f"Feature list check  : {len(_meta['feature_cols'])} features match current FEATURE_COLS: {len(_meta['feature_cols']) == len(FEATURE_COLS)}")
print(f"Forecast CSV check  : {len(pd.read_csv('forecast_output.csv'))} rows")
